# 03 · Join Sofascore + Capology — Italy Serie A 23/24

Integración de estadísticas de rendimiento (Sofascore) con datos salariales (Capology)
para la temporada **2023/24 de Serie A italiana**.

**Flujo de matching:**
1. Normalización de nombres (tildes, mayúsculas, caracteres especiales)
2. `TEAM_MAP`: alineación manual de nombres de equipo entre fuentes
3. Merge exacto normalizado
4. Fuzzy matching en cuatro niveles:
   - Score ≥ 0.90 → aceptación automática
   - 0.75 ≤ score < 0.90 → revisión manual
   - 0.50 ≤ score < 0.75 → revisión manual estricta
   - score < 0.50 → revisión manual muy estricta
5. Revisión de jugadores sin salario
6. Guardado en `data/master/`

---

## 1. Imports y rutas

In [1]:
import pandas as pd
import unicodedata
import re
from rapidfuzz import fuzz
from pathlib import Path
from IPython.display import display

ROOT       = Path.cwd().parents[1]
SF_DIR     = ROOT / 'data' / 'processed' / 'sofascore'
CG_DIR     = ROOT / 'data' / 'processed' / 'capology'
MASTER_DIR = ROOT / 'data' / 'master'
MASTER_DIR.mkdir(parents=True, exist_ok=True)

print('✅ Rutas configuradas')
print(f'   Root:   {ROOT}')
print(f'   Master: {MASTER_DIR}')

✅ Rutas configuradas
   Root:   d:\USER\Desktop\TFM
   Master: d:\USER\Desktop\TFM\data\master


## 2. Función de normalización

In [2]:
def normalize(s):
    """
    Normaliza un string para comparación: elimina tildes, pasa a minúsculas,
    elimina caracteres especiales y espacios extra.
    """
    if pd.isna(s):
        return ''
    s = str(s)
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('ascii')
    s = re.sub(r'[^a-z0-9\s]', ' ', s.lower().strip())
    return re.sub(r'\s+', ' ', s).strip()

print('✅ Función definida')

✅ Función definida


## 3. Carga de datos

In [3]:
df_sf = pd.read_csv(SF_DIR / 'df_italy_2324.csv').copy()
df_cg = pd.read_csv(CG_DIR / 'cg_italy_2324.csv').copy()

print(f'Sofascore:  {df_sf.shape[0]} jugadores | {df_sf.shape[1]} columnas')
print(f'Capology:   {df_cg.shape[0]} jugadores | {df_cg.shape[1]} columnas')

Sofascore:  590 jugadores | 116 columnas
Capology:   659 jugadores | 9 columnas


## 4. Normalización

In [4]:
df_sf['player_norm'] = df_sf['player'].apply(normalize)
df_sf['team_norm']   = df_sf['team'].apply(normalize)
df_cg['player_norm'] = df_cg['player'].apply(normalize)
df_cg['team_norm']   = df_cg['club'].apply(normalize)

print('✅ Normalización aplicada')

✅ Normalización aplicada


## 5. Alineación de equipos (TEAM_MAP)

### 5.1 Identificar discrepancias de nombres de equipo

In [5]:
solo_sf = set(df_sf['team_norm'].unique()) - set(df_cg['team_norm'].unique())
solo_cg = set(df_cg['team_norm'].unique()) - set(df_sf['team_norm'].unique())

print('En Sofascore pero no en Capology:')
for e in sorted(solo_sf): print(f'   {e}')
print()
print('En Capology pero no en Sofascore:')
for e in sorted(solo_cg): print(f'   {e}')

En Sofascore pero no en Capology:
   inter
   milan

En Capology pero no en Sofascore:
   ac milan
   inter milan


### 5.2 Aplicar TEAM_MAP

Rellenar con las discrepancias identificadas en la celda anterior.

In [6]:
# ── Ajustar según la celda anterior ──────────────────────────
TEAM_MAP = {'ac milan':'milan',
            'inter milan':'inter'

}
# ─────────────────────────────────────────────────────────────

df_cg['team_norm'] = df_cg['team_norm'].replace(TEAM_MAP)

diff = set(df_cg['team_norm'].unique()) - set(df_sf['team_norm'].unique())
if diff:
    print(f'⚠️  Equipos de CG aún sin match en SF: {diff}')
else:
    print('✅ Todos los equipos alineados')


✅ Todos los equipos alineados


## 6. Merge exacto normalizado

In [7]:
df_merged = df_sf.merge(
    df_cg[['player_norm', 'team_norm', 'gross_weekly_eur', 'gross_annual_eur',
            'position', 'age', 'nationality']],
    on=['player_norm', 'team_norm'],
    how='left'
)

matched = df_merged['gross_annual_eur'].notna().sum()
total   = len(df_merged)

print(f'Merge exacto: {matched}/{total} ({matched/total:.1%})')
print(f'Sin emparejar: {total - matched}')

Merge exacto: 538/590 (91.2%)
Sin emparejar: 52


## 7. Fuzzy matching sobre los no emparejados

Se generan candidatos para todos los jugadores sin match exacto,
sin umbral mínimo, y se clasifican en cuatro niveles.

In [8]:
df_unmatched = df_merged[df_merged['gross_annual_eur'].isna()].copy()
cg_by_team   = df_cg.groupby('team_norm')['player_norm'].apply(list).to_dict()

rows = []
for _, row in df_unmatched[['player','team','player_norm','team_norm']].drop_duplicates().iterrows():
    candidates = cg_by_team.get(row['team_norm'], [])
    best_match, best_score = None, 0
    for cand in candidates:
        score = fuzz.ratio(row['player_norm'], cand) / 100
        if score > best_score:
            best_score = score
            best_match = cand
    if best_match is not None:
        rows.append({
            'player_sf'  : row['player'],
            'team'       : row['team'],
            'player_norm': row['player_norm'],
            'team_norm'  : row['team_norm'],
            'cg_match'   : best_match,
            'score'      : round(best_score, 3)
        })

df_candidates = pd.DataFrame(rows).sort_values('score', ascending=False)
auto_matches     = df_candidates[df_candidates['score'] >= 0.90].copy()
review_matches   = df_candidates[(df_candidates['score'] >= 0.75) & (df_candidates['score'] < 0.90)].copy()
low_matches      = df_candidates[(df_candidates['score'] >= 0.50) & (df_candidates['score'] < 0.75)].copy()
very_low_matches = df_candidates[df_candidates['score'] < 0.50].copy()

print(f'Auto-aceptados    (score ≥ 0.90):          {len(auto_matches)}')
print(f'Revisión media    (0.75 ≤ score < 0.90):   {len(review_matches)}')
print(f'Revisión estricta (0.50 ≤ score < 0.75):   {len(low_matches)}')
print(f'Revisión muy est. (score < 0.50):           {len(very_low_matches)}')

Auto-aceptados    (score ≥ 0.90):          11
Revisión media    (0.75 ≤ score < 0.90):   3
Revisión estricta (0.50 ≤ score < 0.75):   21
Revisión muy est. (score < 0.50):           17


### 7.1 Matches automáticos (score ≥ 0.90)

Revisar para confirmar que todos son correctos.

In [9]:
auto_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
27,Pantelis Hatzidiakos,Cagliari,pantelis chatzidiakos,0.976
2,Albert Guðmundsson,Genoa,albert gudmundsson,0.971
16,Paweł Dawidowicz,Hellas Verona,pawel dawidowicz,0.968
9,Łukasz Skorupski,Bologna,lukasz skorupski,0.968
43,Mateusz Łęgowski,Salernitana,mateusz legowski,0.968
10,Evan Ndicka,Roma,evan n dicka,0.957
4,Aleksei Miranchuk,Atalanta,aleksey miranchuk,0.941
42,Filip Jagiełło,Genoa,filip jagiello,0.923
26,Dani Silva,Hellas Verona,daniel silva,0.909
6,Milan Đurić,Monza,milan djuric,0.909


### 7.2 Revisión media (0.75 ≤ score < 0.90)

Añadir a `EXCLUDE_FROM_FUZZY` el `player_norm` de los incorrectos.

In [10]:
review_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
3,Yann Bisseck,Inter,yann aurel bisseck,0.800
44,Michel Adopo,Atalanta,michel ndary adopo,0.800
21,Marcus Pedersen,Sassuolo,marcus holmgren pedersen,0.769


In [11]:
# ── Falsos positivos a excluir del nivel medio ────────────────
EXCLUDE_FROM_FUZZY = [

]
# ─────────────────────────────────────────────────────────────

review_accepted = review_matches[~review_matches['player_norm'].isin(EXCLUDE_FROM_FUZZY)]
print(f'Aceptados: {len(review_accepted)} | Excluidos: {len(EXCLUDE_FROM_FUZZY)}')


Aceptados: 3 | Excluidos: 0


### 7.3 Revisión estricta (0.50 ≤ score < 0.75)

Por defecto ninguno se acepta. Añadir a `ACCEPT_LOW_FUZZY` los correctos.

In [12]:
low_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
37,Leonardo Cerri,Juventus,leonardo bonucci,0.733
5,Mario Gila Fuentes,Lazio,mario gila,0.714
38,Mamadou Coulibaly,Salernitana,lassana coulibaly,0.706
7,Federico Di Francesco,Lecce,federico brancolini,0.700
17,Andrea Ferraris,Monza,andrea carboni,0.690
35,Vivaldo,Udinese,vivaldo semedo,0.667
8,Frank Anguissa,Napoli,andre zambo anguissa,0.647
25,Davide Bartesaghi,Milan,davide calabria,0.625
41,Nicolás Domínguez,Bologna,nicola bagnolini,0.606
39,Luigi Canotto,Frosinone,pierluigi frattali,0.581


In [13]:
# ── Matches de score bajo confirmados manualmente ─────────────
ACCEPT_LOW_FUZZY = ['mario gila fuentes',
                    'vivaldo',
                    'frank anguissa'

]
# ─────────────────────────────────────────────────────────────

low_accepted = low_matches[low_matches['player_norm'].isin(ACCEPT_LOW_FUZZY)]
print(f'Aceptados del nivel bajo: {len(low_accepted)}')


Aceptados del nivel bajo: 3


### 7.4 Revisión muy estricta (score < 0.50)

Por defecto ninguno se acepta. Añadir a `ACCEPT_VERY_LOW_FUZZY` los correctos.

In [14]:
very_low_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
22,Ola Solbakken,Roma,tommaso baldanzi,0.483
34,João Costa,Roma,bryan cristante,0.480
23,Simone Verdi,Torino,samuele ricci,0.480
24,Giacomo Corona,Empoli,nicolo cambiaghi,0.467
11,Jan-Carlo Simić,Milan,yacine adli,0.462
51,Justin Kumi,Sassuolo,agustin alvarez,0.462
47,Nikola Sekulov,Juventus,matias soule,0.462
18,Ebenezer Akinsanmiro,Inter,benjamin pavard,0.457
20,Kingstone Mutandwa,Cagliari,antoine makoumbou,0.457
49,Joseph Nonge,Juventus,moise kean,0.455


In [15]:
# ── Matches very low confirmados manualmente ──────────────────
ACCEPT_VERY_LOW_FUZZY = [

]
# ─────────────────────────────────────────────────────────────

very_low_accepted = very_low_matches[very_low_matches['player_norm'].isin(ACCEPT_VERY_LOW_FUZZY)]
print(f'Aceptados del nivel very low: {len(very_low_accepted)}')


Aceptados del nivel very low: 0


### 7.5 Aplicar todos los fuzzy matches aceptados

In [16]:
all_fuzzy    = pd.concat([auto_matches, review_accepted, low_accepted, very_low_accepted], ignore_index=True)
fuzzy_lookup = dict(zip(all_fuzzy['player_norm'], all_fuzzy['cg_match']))

df_merged['player_norm_fuzzy'] = df_merged.apply(
    lambda r: fuzzy_lookup.get(r['player_norm'], r['player_norm'])
    if pd.isna(r['gross_annual_eur']) else r['player_norm'],
    axis=1
)

df_final = (
    df_merged
    .drop(columns=['gross_weekly_eur', 'gross_annual_eur', 'position', 'age', 'nationality'])
    .merge(
        df_cg[['player_norm', 'team_norm', 'gross_weekly_eur', 'gross_annual_eur',
               'position', 'age', 'nationality']],
        left_on=['player_norm_fuzzy', 'team_norm'],
        right_on=['player_norm', 'team_norm'],
        how='left'
    )
    .drop(columns=['player_norm_y', 'player_norm_fuzzy'])
    .rename(columns={'player_norm_x': 'player_norm'})
)

matched_final = df_final['gross_annual_eur'].notna().sum()
print(f'Resultado final: {matched_final}/{len(df_final)} ({matched_final/len(df_final):.1%})')
print(f'Sin salario:     {len(df_final) - matched_final}')

Resultado final: 555/590 (94.1%)
Sin salario:     35


## 8. Revisión de jugadores sin salario

Ordenados por equipo y minutos jugados para identificar si alguno debería tener salario.

In [17]:
sin_salario = (
    df_final[df_final['gross_annual_eur'].isna()]
    [['player', 'team', 'minutesPlayed', 'appearances', 'goals', 'assists']]
    .sort_values(['team', 'minutesPlayed'], ascending=[True, False])
    .reset_index(drop=True)
)

pd.set_option('display.max_rows', None)
print(f'Total sin salario: {len(sin_salario)}')
display(sin_salario)
pd.reset_option('display.max_rows')

Total sin salario: 35


,player,team,minutesPlayed,appearances,goals,assists
0,Leonardo Mendicino,Atalanta,1,1,0,0
1,Nicolás Domínguez,Bologna,101,2,0,0
2,Kingstone Mutandwa,Cagliari,52,5,1,0
3,Nicolas Haas,Empoli,90,1,0,0
4,Liam Henderson,Empoli,11,1,0,0
5,Giacomo Corona,Empoli,9,1,0,0
6,Andrea Sodero,Empoli,9,1,0,0
7,Gennaro Borrelli,Frosinone,23,1,0,0
8,Luigi Canotto,Frosinone,15,1,0,0
9,Przemysław Szymiński,Frosinone,8,1,0,0


### 8.1 Comparación manual por equipo

Para cada equipo con jugadores sin salario se muestra la plantilla completa de Capology
ordenada alfabéticamente por nombre normalizado, facilitando la detección visual de matches fallidos.

In [18]:
equipos_sin_salario = sin_salario['team'].unique()

for equipo in sorted(equipos_sin_salario):
    sf_jugadores = sin_salario[sin_salario['team'] == equipo][['player', 'minutesPlayed']].sort_values('player')

    equipo_norm  = normalize(equipo)
    cg_jugadores = (
        df_cg[df_cg['team_norm'] == equipo_norm][['player', 'player_norm']]
        .sort_values('player_norm')
        .reset_index(drop=True)
    )

    print(f'\n{"="*60}')
    print(f'  {equipo}  —  SF sin salario:')
    display(sf_jugadores.reset_index(drop=True))
    print(f'  CG plantilla completa:')
    display(cg_jugadores)


  Atalanta  —  SF sin salario:


,player,minutesPlayed
0,Leonardo Mendicino,1


  CG plantilla completa:


,player,player_norm
0,Ademola Lookman,ademola lookman
1,Aleksey Miranchuk,aleksey miranchuk
2,Berat Djimsiti,berat djimsiti
3,Charles De Ketelaere,charles de ketelaere
4,Davide Zappacosta,davide zappacosta
5,Éderson,ederson
6,El Bilal Touré,el bilal toure
7,Emil Holm,emil holm
8,Francesco Rossi,francesco rossi
9,Gianluca Scamacca,gianluca scamacca



  Bologna  —  SF sin salario:


...

,player,player_norm
0,Adam Masina,adam masina
1,Adrien Tamèze,adrien tameze
2,Alessandro Buongiorno,alessandro buongiorno
3,Ange Caumenan N'Guessan,ange caumenan n guessan
4,Antonio Sanabria,antonio sanabria
5,Brandon Soppy,brandon soppy
6,David Okereke,david okereke
7,David Zima,david zima
8,Demba Seck,demba seck
9,Duván Zapata,duvan zapata



  Udinese  —  SF sin salario:


,player,minutesPlayed
0,Beto,74


  CG plantilla completa:


,player,player_norm
0,Adam Masina,adam masina
1,Antonio Tikvic,antonio tikvic
2,Axel Guessand,axel guessand
3,Brenner,brenner
4,Christian Kabasele,christian kabasele
5,Daniele Padelli,daniele padelli
6,David Pejičić,david pejicic
7,Domingos Quina,domingos quina
8,Enzo Ebosse,enzo ebosse
9,Etienne Camara,etienne camara


In [19]:
# ── Matches manuales (nombres muy distintos o traspasos invernales) ──
# Formato: (player_norm_sf, team_norm_sf): (player_norm_cg, team_norm_cg)
MANUAL_MATCHES = {

}
# ────────────────────────────────────────────────────────────────────
print(f'Matches manuales definidos: {len(MANUAL_MATCHES)}')


Matches manuales definidos: 0


In [20]:
# Aplicar matches manuales sobre los que siguen sin salario
for (p_sf, t_sf), (p_cg, t_cg) in MANUAL_MATCHES.items():
    mask = (df_final['player_norm'] == p_sf) & (df_final['team_norm'] == t_sf) & (df_final['gross_annual_eur'].isna())
    datos_cg = df_cg[(df_cg['player_norm'] == p_cg) & (df_cg['team_norm'] == t_cg)]
    if not datos_cg.empty and mask.any():
        for col in ['gross_weekly_eur', 'gross_annual_eur', 'position', 'age', 'nationality']:
            df_final.loc[mask, col] = datos_cg[col].values[0]
        print(f'✅ Match manual aplicado: {p_sf} ({t_sf}) → {p_cg} ({t_cg})')
    else:
        print(f'⚠️  No encontrado: {p_sf} ({t_sf}) → {p_cg} ({t_cg})')

matched_tras_manual = df_final['gross_annual_eur'].notna().sum()
print(f'\nTras matches manuales: {matched_tras_manual}/{len(df_final)} ({matched_tras_manual/len(df_final):.1%})')


Tras matches manuales: 555/590 (94.1%)


In [21]:
pd.reset_option('display.max_rows')

## 9. Guardado

Una vez revisado todo, se eliminan las columnas auxiliares y se guarda en `data/master/`.

In [22]:
df_final = df_final.drop(columns=['player_norm', 'team_norm'])

nombre_salida = 'master_italy_2324.csv'
df_final.to_csv(MASTER_DIR / nombre_salida, index=False)

print(f'✅ Guardado: {nombre_salida}')
print(f'   Jugadores totales:  {len(df_final)}')
print(f'   Con salario:        {df_final["gross_annual_eur"].notna().sum()}')
print(f'   Sin salario (NaN):  {df_final["gross_annual_eur"].isna().sum()}')
print(f'   Columnas:           {df_final.shape[1]}')

✅ Guardado: master_italy_2324.csv
   Jugadores totales:  590
   Con salario:        555
   Sin salario (NaN):  35
   Columnas:           121
